# 歌词采集-QQ音乐
不需要按专辑采集

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter


In [2]:
import sys

sys.path.append('..')

from data_crawler import  get_songs_data_raw, get_all_songs_lyric, clear_and_save_lyric
from songs_libs import format_timestamp, SongDataCleaner

# main

In [66]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

# file_path_prefix = "data/liyuchun/"
# singer = "李宇春"
# max_page = 14

file_path_prefix = "data/sunyanzi/"
singer = "孙燕姿"
max_page = 10

### 曲目采集

In [67]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...
正在获取第4页数据...
正在获取第5页数据...
正在获取第6页数据...
正在获取第7页数据...
正在获取第8页数据...
正在获取第9页数据...
正在获取第10页数据...


In [68]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [69]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [70]:
df_song_data_raw_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time
0,5211338,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,001pWERg3vFgg8,逆光,15902,000P3l050Olt27,289,1174492800
1,5646,003PSWcB4EDl6P,开始懂了,NaN,孙燕姿,109,001pWERg3vFgg8,我要的幸福,493,004KRIQr1oCp2A,271,976118400
2,8145,001xJbdb2MBaaz,遇见,《向左走，向右走》电影主题曲,孙燕姿,109,001pWERg3vFgg8,The Moment,586,002ehzTm0TxXC2,210,1061481600
3,100893820,001pHzdz29aG1Z,雨天,NaN,孙燕姿,109,001pWERg3vFgg8,My Story 2006 新歌+精选,837815,000VMnoi2Lgt44,239,1159200000
4,213446637,0042ih853gboG4,半句再见,"From ""At Café 6"" / Main Theme Song",孙燕姿,109,001pWERg3vFgg8,半句再见,3972762,003c8Hsp0OrnXY,242,1522339200
...,...,...,...,...,...,...,...,...,...,...,...,...
495,322268981,004fEDt63ria0t,矜持,NaN,孙燕姿,109,001pWERg3vFgg8,NaN,0,NaN,189,0
496,345487411,000LgW6E2RBKJ9,遇见 (2003华纳十周年演唱会),NaN,孙燕姿,109,001pWERg3vFgg8,NaN,0,NaN,219,1061481600
497,125351768,000CUwy34ISlvb,Venus (恰恰版),NaN,孙燕姿,109,001pWERg3vFgg8,NaN,0,NaN,244,0
498,433024134,002eDAZn1r4XHu,Honey Honey-孙燕姿-ringtone,NaN,孙燕姿,0,0032fmHO2UDnV3,NaN,0,NaN,51,0


### 清洗

In [71]:
albums_to_delete = ['声生不息', '我歌', '中国梦', '谁是大歌神', '梦想的声音', '我是歌手', 'JJ的咖啡调调']

In [72]:
df_songs = SongDataCleaner.clear_song_name(df_song_data_raw_read)
df_songs = SongDataCleaner.clear_song_singer(df_songs, singer)
df_songs = SongDataCleaner.clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,5211338,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,001pWERg3vFgg8,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,我怀念的,逆光
1,5646,003PSWcB4EDl6P,开始懂了,NaN,孙燕姿,109,001pWERg3vFgg8,我要的幸福,493,004KRIQr1oCp2A,271,976118400,开始懂了,开始懂了,我要的幸福
2,8145,001xJbdb2MBaaz,遇见,《向左走，向右走》电影主题曲,孙燕姿,109,001pWERg3vFgg8,The Moment,586,002ehzTm0TxXC2,210,1061481600,遇见,遇见,The Moment
3,100893820,001pHzdz29aG1Z,雨天,NaN,孙燕姿,109,001pWERg3vFgg8,My Story 2006 新歌+精选,837815,000VMnoi2Lgt44,239,1159200000,雨天,雨天,My Story 2006 新歌+精选
4,213446637,0042ih853gboG4,半句再见,"From ""At Café 6"" / Main Theme Song",孙燕姿,109,001pWERg3vFgg8,半句再见,3972762,003c8Hsp0OrnXY,242,1522339200,半句再见,半句再见,半句再见
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,406077345,000zavzx3OOwnJ,残酷な天使のテーゼ,NaN,孙燕姿,0,0032fmHO2UDnV3,NaN,0,NaN,243,0,残酷な天使のテーゼ,残酷な天使のテーゼ,nan
192,614820957,001Bacc828H8XM,雨天 - 铃声,NaN,孙燕姿,0,0032fmHO2UDnV3,NaN,0,NaN,41,0,雨天-铃声,雨天-铃声,nan
193,613466364,001Muh9C4IqXDJ,开始懂了 - 铃声版,NaN,孙燕姿,0,0032fmHO2UDnV3,NaN,0,NaN,48,0,开始懂了-铃声版,开始懂了-铃声版,nan
194,322268981,004fEDt63ria0t,矜持,NaN,孙燕姿,109,001pWERg3vFgg8,NaN,0,NaN,189,0,矜持,矜持,nan


In [73]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()


## 歌单确认

In [74]:
# 李宇春
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']
# 陈奕迅
songs_to_delete = ['新曲+精选', 'K歌之王AIR', '慢慢喜欢你', '最冷一天']
# 任贤齐
songs_to_delete = ['伤心太平洋+心太软+我是一只鱼+对面的女孩看过来', '桥边姑娘', '你知道我在等你吗', '我是一只小小鸟', '外婆的澎湖湾2015', '海阔天空', '爱的路上只有你和我']
# 林俊杰
songs_to_delete = ['开场白', '无聊']
songs_list_final = songs_list_130.copy()

In [ ]:
# 添加
for i in songs_to_add:
    if i not in songs_list_final:
        print(i)
        songs_list_final.append(i)

In [62]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

开场白
无聊


In [63]:
len(songs_list_final)

128

## 歌曲数据确认

In [75]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,5211338,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,001pWERg3vFgg8,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
1,5646,003PSWcB4EDl6P,开始懂了,NaN,孙燕姿,109,001pWERg3vFgg8,我要的幸福,493,004KRIQr1oCp2A,271,976118400,开始懂了,我要的幸福,开始懂了,2000-12-07,2000
2,8145,001xJbdb2MBaaz,遇见,《向左走，向右走》电影主题曲,孙燕姿,109,001pWERg3vFgg8,The Moment,586,002ehzTm0TxXC2,210,1061481600,遇见,The Moment,遇见,2003-08-22,2003
3,100893820,001pHzdz29aG1Z,雨天,NaN,孙燕姿,109,001pWERg3vFgg8,My Story 2006 新歌+精选,837815,000VMnoi2Lgt44,239,1159200000,雨天,My Story 2006 新歌+精选,雨天,2006-09-26,2006
4,213446637,0042ih853gboG4,半句再见,"From ""At Café 6"" / Main Theme Song",孙燕姿,109,001pWERg3vFgg8,半句再见,3972762,003c8Hsp0OrnXY,242,1522339200,半句再见,半句再见,半句再见,2018-03-30,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,491752,004JJW4v2So2tx,作战,NaN,孙燕姿,109,001pWERg3vFgg8,Leave,39480,0033sQMp2BZDJC,218,1021910400,作战,Leave,作战,2002-05-21,2002
126,5211227,002lRwn62glwFS,终于,NaN,孙燕姿,109,001pWERg3vFgg8,孙燕姿同名专辑,8707,002UZ9ob4Ecg0S,270,960393600,终于,孙燕姿同名专辑,终于,2000-06-08,2000
127,491779,003rGQse4RcMpr,Silent All These Years,NaN,孙燕姿,109,001pWERg3vFgg8,Start 自选集,39482,000FAIFd0r22m9,255,1012492800,SilentAllTheseYears,Start 自选集,SilentAllTheseYears,2002-02-01,2002
128,5187,000sPOPw1FgNDK,种,NaN,孙燕姿,109,001pWERg3vFgg8,Stefanie,451,003CS0lX1DwEcN,251,1098979200,种,Stefanie,种,2004-10-29,2004


In [76]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [77]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

我怀念的
开始懂了
遇见
雨天
半句再见
天黑黑
我不难过
眼泪成诗
逆光
绿光
原来你什么都不要
当冬夜渐暖
第一天
日落
克卜勒
Honey Honey
样子
坏天气
180度
愚人的国度
The Moment
隐形人
逃亡
我也很想他
Hey Jude
咕叽咕叽
同类
漩涡
我的爱
极美
直来直往
风衣
在,也不见
飘着
神奇
超快感
我要的幸福
尚好的青春
爱情证书
银泰
不是真的爱我
爱情字典
天使的指纹
比较幸福
任性
安宁
了解
平日快乐
一样的夏天
无限大
星期一天气晴我离开你
相信
完美的一天
需要你
天天年年
Venus
时光小偷
祝你开心
风筝
奔
永远
和平
雨还是不停地落下
渴
明天的记忆
余额
世界终结前一天
休止符
眼神
跳舞的梵谷
这个世界
懂事
心愿
害怕
年轻无极限
E-Lover
不能和你一起
很好
彩虹金刚
一起走到
错觉
我想
我很愉快
追
懒得去管
天越亮，夜越黑
梦游
木兰情
世说心语
Radio
Stefanie
我不爱
流浪地图
守护永恒的爱
爱从零开始
天空
学会
难得一见
没有人的方向
关于
真的
是时候 + Hidden Track
慢慢来
Leave Me Alone
围绕
接下来
未完成
浓眉毛
听见
明天晴天
练习
零缺点
太阳底下
另一张脸
随堂测验
橄榄树
累赘
空口言
梦不落
充氧期
Sometimes Love Just Ain't Enough
未知的精彩
不同
中间地带
Leave
作战
终于
Silent All These Years
种
超人类


## 歌词清洗

In [78]:
clear_and_save_lyric(file_path_prefix, df_songs_final)

In [79]:
# 歌词数据查验
df_lyric = pd.read_json(f"{file_path_prefix}cleared_lyric_data.json")
df_lyric

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text
0,5211338,我怀念的,2026-02-24 00:16:45,1,姚若龙,李偲菘,Martin Tang,我问为什么。那女孩传简讯给我。而你为什么。不解释低着头沉默。我该相信你很爱我。不愿意敷衍我。...
1,5646,开始懂了,2026-02-24 00:21:51,1,姚若龙,李偲菘,Kenn C,我竟然没有调头。最残忍那一刻。静静看你走。一点都不像我。原来人会变得温柔。是透彻的懂了。爱情...
2,8145,遇见,2026-02-24 00:24:45,1,易家扬,林一峰,Terence Teo,听见，冬天的离开。我在某年某月，醒过来。我想，我等，我期待。未来却不能因此安排。阴天，傍晚，...
3,100893820,雨天,2026-02-24 00:17:51,1,小寒,李伟菘,胡雪阳/李偲菘,站在十字路的交点。该怎么走，我却只想回头。除了你给的伞，我再也没有。别的借口，去拥有你的什么...
4,213446637,半句再见,2026-02-24 00:17:03,1,藤井树,李偲菘,Terence Teo,一张照片，半句再见。尘封的纪念。用眼泪把你复习一遍。残缺的诗篇，遗忘的誓言。谁脑海有张忘不掉...
...,...,...,...,...,...,...,...,...
125,491752,作战,2026-02-24 00:10:34,1,廖莹如,包小柏,钟兴民,It&apos;s，my，life。我看到你跟自己作战。想不通就把念头整个翻过来。我跟自己开...
126,5211227,终于,2026-02-24 00:21:58,1,易家扬,李伟菘,吴庆隆,努力等着你。我很小心，偷偷待在你世界里。你不会知道的。某一些夜里。偶而你会提起埋在你心里的过...
127,491779,Silent All These Years,2026-02-24 00:08:21,1,,,,Excuse，me，but，can，I，be，you，for，a，while。My，dog，...
128,5187,种,2026-02-24 00:35:35,1,廖莹如,孙燕姿,Kenn C,某个时候我们一起低头。一块地是你的。想想将来它会成长出什么，啊。新鲜水果还是一栋大楼。梦想现...


In [82]:
df_lyric['lyric_length'] = df_lyric['lyrics_text'].apply(lambda x: len(x))
df_lyric.sort_values(by='lyric_length')

,song_id,song_name,start_time,has_lyric,lyricist,composer,arranger,lyrics_text,lyric_length
115,491780,橄榄树,2026-02-24 00:22:49,1,三毛,李泰祥,,不要问我从哪里来。我的故乡在远方。为什么流浪。流浪远方，流浪。为了天空飞翔的小鸟。为了山间轻...,159
95,491787,天空,2026-02-24 00:00:55,1,黄桂兰,杨明煌,,我的天空，为何挂满湿的泪。我的天空，为何总灰的脸。飘流在世界的另一边。任寂寞侵犯一遍一遍。天...,196
48,491756,一样的夏天,2026-02-24 00:23:01,1,JOY,陈达伟,陈达伟,窗外的雨刚刚停。午后气息浓浓地才散去。迷迷糊糊张开眼。刚刚的梦。我似乎在瞬间看见你。Oh，m...,209
87,643383,木兰情,2026-02-24 00:29:06,1,易家扬,李偲菘,,我看得见云在天上混乱地飞。我听得见滚滚沙场埋一滴泪。这是谁的沙漠。我忘了我是谁。又是谁，让这...,229
82,206621750,我很愉快,2026-02-24 00:30:12,1,吕康惟,吕康惟,Kenn C,捡起地上的谎。满足了欲望。其实美好。不难啊。一个人闷到慌。也不想作罢。谁叫我们。那么爱啊。请...,237
...,...,...,...,...,...,...,...,...,...
9,5651,绿光,2026-02-24 00:02:58,1,天天,李偲菘,Kenn C,Green，light's，right，here。身边，身边。期待着一个幸运和一个冲击。多么...,895
55,491785,Venus,2026-02-24 00:22:06,1,,,,Goddess，On，The，Mountain，Top。Burning，Like，a，Sil...,1052
24,491778,Hey Jude,2026-02-24 00:05:52,1,,,,Hey，Jude，don't，make，it，bad。Take，a，sad，song，and...,1187
120,491784,Sometimes Love Just Ain't Enough,2026-02-24 00:26:05,1,,,,I，don't，wanna，lose，you'。I，don't，wanna，use，you。...,1432
